# LUAD trial matcher: Phase 2 evaluation on Colab (T4 GPU)

Runs `luad-match retrieve -> evaluate -> rank` with **phi4 (14B) served by Ollama on the Colab GPU**.

Before running:
- **Runtime -> Change runtime type -> T4 GPU.**
- Put `trials_parsed.json` and a patient JSON in the Drive folder set below
  (create the example patient locally with `luad-match example-patient --out patient_example.json`).
- Optional: add `ANTHROPIC_API_KEY` under Colab **Secrets** (key icon) to enable API re-queries
  for criteria with local confidence in [0.35, 0.75]. Without it those criteria keep the local verdict
  and are marked `route = api_unavailable`.

Results and the LLM call log are written straight to Drive after every trial, so a disconnected
session can be resumed by re-running the cells (`--resume`).

In [ ]:
# ---- Configuration ----------------------------------------------------------
REPO_URL = "https://github.com/AlwazH0ney/trial-matcher.git"
BRANCH = "main"

DRIVE_DIR = "/content/drive/MyDrive/trial-matcher"   # holds the inputs below
TRIALS_FILE = "trials_parsed.json"
PATIENT_FILE = "patient_example.json"
RUN_NAME = "phi4_colab_t4"                            # results go to DRIVE_DIR/results/RUN_NAME

TOP_K = 50                # candidates passed from retrieval to the LLM
OLLAMA_MODEL = "phi4"     # 14B; ~9 GB download
AS_OF = None              # e.g. "2026-09-30"; None = today
REPO_DIR = "/content/trial-matcher"

In [ ]:
import os, shutil, subprocess, sys, time, json, urllib.request
from pathlib import Path

def run(cmd, cwd=None, env=None):
    """Run a shell command, stream its output, and raise if it fails."""
    print(f"$ {cmd}", flush=True)
    proc = subprocess.Popen(cmd, shell=True, cwd=cwd, env=env, text=True,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    for line in proc.stdout:
        print(line, end="", flush=True)
    if proc.wait() != 0:
        raise RuntimeError(f"command failed with exit code {proc.returncode}: {cmd}")

gpu = subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader",
                     shell=True, capture_output=True, text=True).stdout.strip()
if not gpu:
    raise RuntimeError("No GPU attached. Runtime -> Change runtime type -> T4 GPU, then re-run.")
print("GPU:", gpu)

## 1. Clone the repo and install the package

In [ ]:
if Path(REPO_DIR, ".git").exists():
    run(f"git -C {REPO_DIR} fetch --depth 1 origin {BRANCH} && git -C {REPO_DIR} checkout -q FETCH_HEAD")
else:
    run(f"git clone --depth 1 -b {BRANCH} {REPO_URL} {REPO_DIR}")
os.chdir(REPO_DIR)
run("pip install -q -e .")
run("luad-match --version")

## 2. Install and start Ollama, pull phi4

In [ ]:
# zstd: the installer unpacks a .tar.zst; pciutils: lets it detect the NVIDIA GPU.
run("apt-get -qq update && apt-get -qq install -y zstd pciutils > /dev/null")
if not shutil.which("ollama"):
    run("curl -fsSL https://ollama.com/install.sh | sh")
run("ollama --version || true")

In [ ]:
# Start the server in the background; the matcher reaches it through OLLAMA_HOST.
os.environ["OLLAMA_HOST"] = "127.0.0.1:11434"
os.environ["OLLAMA_KEEP_ALIVE"] = "30m"
!nohup ollama serve > /content/ollama.log 2>&1 &

for _ in range(60):
    try:
        urllib.request.urlopen("http://127.0.0.1:11434/api/tags", timeout=2)
        print("Ollama is up at", os.environ["OLLAMA_HOST"])
        break
    except Exception:
        time.sleep(1)
else:
    print(open("/content/ollama.log").read()[-3000:])
    raise RuntimeError("Ollama did not start within 60 s; see /content/ollama.log above.")

In [ ]:
run(f"ollama pull {OLLAMA_MODEL}")
# Warm-up call: loads the weights onto the GPU so the first criterion's latency is not an outlier.
run(f"ollama run {OLLAMA_MODEL} 'Reply with OK.'")
run("ollama ps")   # PROCESSOR column should read 100% GPU

## 3. Mount Drive and sync inputs into `data/processed/`

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

src = Path(DRIVE_DIR)
dst = Path(REPO_DIR, "data", "processed")
dst.mkdir(parents=True, exist_ok=True)
for name in (TRIALS_FILE, PATIENT_FILE):
    if not (src / name).exists():
        raise FileNotFoundError(f"{src / name} not found. Upload it to {DRIVE_DIR} first.")
    shutil.copy2(src / name, dst / name)
    print("synced", src / name, "->", dst / name)

RESULTS_DIR = src / "results" / RUN_NAME
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print("results ->", RESULTS_DIR)

# Optional API re-query backend
try:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
    print("ANTHROPIC_API_KEY loaded from Colab secrets: API re-query enabled")
except Exception:
    os.environ.pop("ANTHROPIC_API_KEY", None)
    print("No ANTHROPIC_API_KEY secret: mid-confidence criteria will be marked api_unavailable")

## 4. Retrieve candidates, evaluate, rank

In [ ]:
patient = f"data/processed/{PATIENT_FILE}"
candidates = RESULTS_DIR / "candidates.json"
evaluated = RESULTS_DIR / "evaluated.json"
log_path = RESULTS_DIR / "llm_log.jsonl"
report = RESULTS_DIR / "matches.md"

run(f"luad-match retrieve --patient {patient} --trials data/processed/{TRIALS_FILE} "
    f"--top-k {TOP_K} --out '{candidates}' --index-dir data/processed/index")

In [ ]:
# Writes to Drive after every trial. If the session drops, re-run from the top; --resume skips done trials.
as_of = f"--as-of {AS_OF}" if AS_OF else ""
run(f"luad-match evaluate --patient {patient} --candidates '{candidates}' "
    f"--out '{evaluated}' --log '{log_path}' --ollama-model {OLLAMA_MODEL} {as_of} --resume")

In [ ]:
run(f"luad-match rank --evaluated '{evaluated}' --out '{report}' --top 10")

## 5. Write results back to Drive and summarise

In [ ]:
# evaluated.json, llm_log.jsonl, candidates.json and matches.md were written directly to Drive.
# Also keep the Ollama server log and a copy of the report inside the repo checkout.
shutil.copy2("/content/ollama.log", RESULTS_DIR / "ollama.log")
Path(REPO_DIR, "reports").mkdir(exist_ok=True)
shutil.copy2(report, Path(REPO_DIR, "reports", "matches.md"))
for p in sorted(RESULTS_DIR.iterdir()):
    print(f"{p.stat().st_size / 1024:9.1f} KB  {p}")

In [ ]:
ev = json.loads(Path(evaluated).read_text())
s = ev["summary"]
rows = [json.loads(l) for l in Path(log_path).read_text().splitlines() if l.strip()]
local_rows = [r for r in rows if r["backend"] == "local" and not r.get("error")]

print(f"Local model            : {ev['backends']['local']['model']} "
      f"@ {ev['backends']['local'].get('runtime', {}).get('gpu')}")
print(f"API model              : {(ev['backends']['api'] or {}).get('model', 'disabled')}")
print(f"Trials evaluated       : {s['n_trials']}")
print(f"Total criteria         : {s['n_criteria']}")
print(f"Avg time per criterion : {s['mean_criterion_time_s']} s "
      f"(local phi4 only: {s['mean_local_latency_s']} s)")
print(f"Routing rate to API    : {s['api_routing_rate']:.1%} "
      f"({s['n_routed_api']} re-queried; {s['n_api_unavailable']} in band but API disabled; "
      f"band rate {s['routing_band_rate']:.1%})")
print(f"API flipped local      : {s['n_api_flipped']}")
print(f"Unresolved trials      : {s['n_unresolved_trials']} of {s['n_trials']} "
      f"(status counts: {s['n_trials_by_status']})")
print(f"Unknown criteria       : {s['n_unknown_criteria']}")
print(f"Parse errors           : {s['n_parse_errors']}")
print(f"Tokens (local in/out)  : {s['input_tokens']['local']:,} / {s['output_tokens']['local']:,}")
print(f"LLM log rows           : {len(rows)} ({len(local_rows)} local ok)")
print(f"Wall time              : {s['wall_time_s'] / 60:.1f} min")

In [ ]:
from IPython.display import Markdown, display
display(Markdown(Path(report).read_text()))